# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, exploration, and processing of the FAIR^2 dataset using the `mlcroissant` library, referencing all data entities by their `@id` fields.

### Dataset Source
The dataset is described via a Croissant schema URL and supports machine-actionable metadata for reproducible data workflows.

In [ ]:
# Ensure `mlcroissant` is installed (uncomment the next line if running in a new environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and display high-level info
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # mlcroissant returns a pydantic model, not a dict
print(f"{metadata.name}: {metadata.description}")
print(f"\nDataset Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review record sets (tables), their fields (columns), and all referenced `@id` values.

We will print all record set `@id`s, then for each, print their corresponding field (column) `@id`s.

In [ ]:
# List all record sets, their @ids, and fields
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in dataset.")
else:
    print(f"{len(record_sets)} record set(s) found:")
    for rs in record_sets:
        print(f"- RecordSet name: {rs.name}")
        print(f"  @id: {rs.id}")
        if rs.fields:
            print(f"  Fields:")
            for f in rs.fields:
                print(f"    - name: {f.name}\n      @id: {f.id} (dataType: {getattr(f, 'data_type', None)})")
        print("")

## 3. Data Extraction
Load records from a specific record set into a DataFrame. We will extract all available record sets and reference them exclusively by their `@id`s.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))  # Each record is always a dict keyed by field @id
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[rs_id])} records.")
    else:
        print("No records available.")

# Preview first DataFrame if any
if dataframes:
    first_rs_id = record_set_ids[0]
    print(f"\nAvailable fields in record set @id '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head(3))
else:
    print("No DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing, grouping, and handling numeric fields. All field references use their `@id`s.

We'll select a numeric field from the first record set for demonstration. Please replace the example field `@id` if needed by inspecting the output above.

In [ ]:
# Use the first record set as the main analytical table
main_rs_id = record_set_ids[0]
df = dataframes[main_rs_id]

# Find a numeric field
numeric_field = None
for f in dataset.record_set(main_rs_id).fields:
    # Choose first field with type Integer or Float
    if getattr(f, "data_type", None) in ["schema:Integer", "schema:Float", "Integer", "Float"]:
        numeric_field = f.id
        print(f"Selected numeric field: {numeric_field}")
        break

if numeric_field is None:
    raise ValueError("No numeric field detected in this record set.")

threshold = 10
if numeric_field in df.columns:
    # Clean non-numeric values if necessary
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping on a categorical field
    group_field = None
    for f in dataset.record_set(main_rs_id).fields:
        if getattr(f, "data_type", None) in ["schema:Text", "Text", "schema:Boolean"]:
            group_field = f.id
            print(f"Selected group field: {group_field}")
            break

    if group_field and group_field in filtered_df.columns:
        # Only group by group_field if it's not too unique
        if filtered_df[group_field].nunique() < 20:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print(f"group_field '{group_field}' has too many unique values to group meaningfully.")
    else:
        print("No suitable categorical group field found for grouping.")
else:
    print(f"Numeric field '{numeric_field}' not found in DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships. We'll plot a histogram of the numeric field and, if applicable, a boxplot grouped by the detected categorical field, all by referencing IDs.

In [ ]:
if numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    if 'group_field' in locals() and group_field and group_field in df.columns:
        # Only plot if group_field has reasonable number of unique values
        if df[group_field].nunique() < 20:
            plt.figure(figsize=(9, 5))
            sns.boxplot(x=df[group_field], y=df[numeric_field])
            plt.title(f"{numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
else:
    print(f"Numeric field '{numeric_field}' not available for plotting.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using the `mlcroissant` library by systematically referencing all entities via their `@id` fields. We:
- Loaded metadata and reviewed available record sets and their fields by `@id`.
- Extracted data from each record set directly into pandas DataFrames with columns keyed by field `@id`s.
- Performed basic EDA including filtering, normalization, and grouping, all via `@id` references.
- Visualized field distributions and relationships using standard Python data science tools.

This workflow ensures full alignment with the Croissant data model and enables robust, reproducible data processing for ML and research.

_For deeper analysis, refer to field descriptions in the Croissant schema, and try cross-referencing multiple record sets using their `@id`s._